# Causal Discovery Analysis

Causal search using BOSS/FGES algorithms via PyTetrad.

**Reference:** Kumu R package, `issue_causal_analysis.Rmd` 

## Notebook Setup Instructions

This guide is for code reviewers to run the notebook end-to-end without additional context.

### Environment Requirements

- **OS:** macOS/Linux/Windows (tested in this repo on macOS).
- **Python:** 3.10+ recommended (repo requirement is `>=3.7`; avoid very old Python).
- **Java:** JDK 11+ recommended (JDK 17 works well for Tetrad-based workflows).
- **Jupyter:** VS Code Notebook or Jupyter Lab.

### Required Python Packages

Install from the repository root (`pykumu/`) in a clean virtual environment:

```bash
python3 -m venv .venv
source .venv/bin/activate  # Windows: .venv\\Scripts\\activate
python -m pip install --upgrade pip
pip install numpy pandas JPype1
```

If `pytetrad` is not already importable, install local package mode from this repo root:

```bash
pip install -e .
```

### Required Files (in `fges_notebook/`)

- `null_variable_dt.csv` (used when `RUN_PREPROCESSING = False`)
- `binarized_variable_dt.csv` (written/used in non-null stage)
- `mike_knowledge_box.txt` (domain-knowledge constraints)
- Optional raw source: `final_ist_cve_smell_interval_dt.csv` (needed only when `RUN_PREPROCESSING = True`)

### Working Directory and Kernel

- Open notebook: `fges_notebook/causal_search_notebook.ipynb`.
- Ensure working directory resolves notebook-relative files (`fges_notebook/`).
- Select the same Python interpreter where dependencies were installed.
- If helper modules change, **restart kernel** before re-running.

### Configuration Checklist (Cell 4)

- `ALGORITHM`: `"boss"` or `"fges"`
- `RUN_PREPROCESSING`:
  - `False` to use existing `null_variable_dt.csv`
  - `True` to regenerate engineered datasets from `RAW_DATA_PATH`
- `KNOWLEDGE_FILE`: verify path exists (default: `mike_knowledge_box.txt`)
- `N_BOOTSTRAP`: lower this for quick validation runs, increase for full analysis

### Recommended Run Order

1. Run **Configuration** and **Import Libraries**.
2. Run **Feature Engineering** (or skip heavy preprocessing by keeping `RUN_PREPROCESSING = False`).
3. Run **FGES Null Variable Search** (produces null-search graph output).
4. Run **Deriving the 1 PNEF Threshold** (`pnef_1` must be created).
5. Run **Non-Null Causal Search**.
6. Run **Applying 1PNEF Threshold**.
7. Run **Results** sections (full graph, subgraph, cycle detection).

### Troubleshooting

- **`AttributeError` on Tetrad methods:** restart kernel and rerun from top (module reload issue).
- **Java heap/memory issues:** reduce `N_BOOTSTRAP`, keep `SHOW_BOOTSTRAP_OUTPUT = False`, rerun.
- **File not found:** verify you are in `fges_notebook/` and required CSV/knowledge files exist.
- **Import errors (`pytetrad`, `jpype`):** confirm the active interpreter matches the environment where packages were installed.

## Configuration

Set algorithm parameters and file paths.

In [1]:
# Analysis Parameters
ALGORITHM = "boss"  # Options: "boss" or "fges"

# Input options
RUN_PREPROCESSING = False  # Set True to recreate engineered datasets from raw timeline CSV
RAW_DATA_PATH = "final_ist_cve_smell_interval_dt.csv"
DATA_PATH = "null_variable_dt.csv"
BINARIZED_DATA_PATH = "binarized_variable_dt.csv"

KNOWLEDGE_FILE = "mike_knowledge_box.txt"
OUTPUT_DIR = "boss_results" if ALGORITHM == "boss" else "fges_results"
NON_NULL_OUTPUT_DIR = "boss_domain_results" if ALGORITHM == "boss" else "fges_domain_results"

# Algorithm-specific parameters
PENALTY_DISCOUNT = 2
N_BOOTSTRAP = 100 if ALGORITHM == "boss" else 50
SEED = 32

# Output control
SHOW_BOOTSTRAP_OUTPUT = False  # Set to True to see bootstrap iteration counts

# 1PNEF Threshold percentile 
PNEF_PERCENTILE = 0.01

# Nodes of interest for sub-graph analysis (effort variables)
NODES_OF_INTEREST = [
    "silence", "silence2",
    "mis_link", "mis_link2",
    "code_dev", "code_dev2",
    "churn", "churn2",
    "commit", "commit2"
]

## Import Libraries

Load required modules for data processing and causal analysis.

In [2]:
# Increase Java memory allocation for large bootstrap analyses
import os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx8g -Xms4g'  # 8GB max heap, 4GB initial
print("✓ Java memory configured: 8GB max, 4GB initial")

✓ Java memory configured: 8GB max, 4GB initial


In [3]:
import pandas as pd
import numpy as np
from pykumu_helpers import (
    load_data, detect_variable_types, 
    parse_graph, prepare_non_null_dataset
 )
from pykumu_algorithms import run_boss_analysis, run_fges_analysis
from pykumu_visualization import (
    prepare_graph_edges, plot_causal_graph, plot_subgraph,
    find_cycles, print_cycle_report
)

Picked up JAVA_TOOL_OPTIONS: -Xmx8g -Xms4g


# Feature Engineering


## Formatting Data Types

### CVE Data Type

In [4]:
if RUN_PREPROCESSING:
    raw_dt = pd.read_csv(RAW_DATA_PATH)
    print(f"Loaded raw data: {raw_dt.shape[0]} rows × {raw_dt.shape[1]} columns")

    # CVE Data Type
    cve_as_str = raw_dt["cve_id"].astype(str)
    last_two_digits_year = cve_as_str.str.slice(6, 8)
    last_four_digits_cve = cve_as_str.str.slice(-4)
    raw_dt["cve_id"] = pd.to_numeric(last_two_digits_year + last_four_digits_cve, errors="coerce")

    commit_interval = raw_dt["commit_interval"].fillna("").astype(str)
    raw_dt["activity_0"] = np.where(commit_interval.eq(""), 1, 0)
    raw_dt["activity_2"] = np.where(commit_interval.ne(""), 1, 0)

    print("Completed: CVE Data Type")

### Convert "start" to Unix Timestamp

In [5]:
if RUN_PREPROCESSING:
    # Convert "start" to Unix Timestamp
    if "start_datetime" in raw_dt.columns:
        raw_dt["start"] = pd.to_datetime(raw_dt["start_datetime"], errors="coerce", utc=True)
    elif "start" in raw_dt.columns:
        raw_dt["start"] = pd.to_datetime(raw_dt["start"], errors="coerce", utc=True)
    else:
        raise ValueError("Expected either 'start_datetime' or 'start' column in raw data")

    raw_dt["start"] = raw_dt["start"].map(lambda x: x.timestamp() if pd.notna(x) else np.nan)
    print("Completed: Convert 'start' to Unix Timestamp")

## Feature Renaming

In [6]:
if RUN_PREPROCESSING:
    # Feature Renaming
    rename_map = {
        "start_datetime": "start",
        "missing_links": "mis_link",
        "radio_silence": "silence",
        "code_only_devs": "code_dev",
        "code_files": "file",
        "ml_only_devs": "mail_dev",
        "ml_threads": "thread",
        "n_commits": "commit"
    }
    available_rename_map = {k: v for k, v in rename_map.items() if k in raw_dt.columns}
    raw_dt = raw_dt.rename(columns=available_rename_map)

    expected_cols = [
        "cve_id", "activity_0", "activity_2", "start",
        "org_silo", "mis_link", "silence", "code_dev", "file",
        "mail_dev", "thread", "commit", "churn"
    ]
    missing_cols = [c for c in expected_cols if c not in raw_dt.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns after formatting: {missing_cols}")

    dt = raw_dt[expected_cols].copy()
    print(f"After Feature Renaming: {dt.shape[0]} rows × {dt.shape[1]} columns")

## Missing Data Handling

In [7]:
if RUN_PREPROCESSING:
    dt["start"] = pd.to_datetime(dt["start"], errors="coerce", utc=True)
    dt = dt[(dt["start"].dt.year < 2000) | (dt["start"].dt.year > 2001)].copy()
    dt["start"] = dt["start"].map(lambda x: x.timestamp() if pd.notna(x) else np.nan)
    dt = dt.fillna(0)
    print(f"After Missing Data Transformations: {dt.shape[0]} rows × {dt.shape[1]} columns")

## 1-Time Lag Features

In [8]:
if RUN_PREPROCESSING:
    lag_feature_cols = [
        "org_silo", "mis_link", "silence", "code_dev", "file",
        "mail_dev", "thread", "commit", "churn"
    ]

    dt = dt.sort_values(["cve_id", "start"]).reset_index(drop=True)

    def add_time_lag(cve_table: pd.DataFrame) -> pd.DataFrame:
        cve_table = cve_table.copy()
        if len(cve_table) < 2:
            for col in lag_feature_cols:
                cve_table[f"{col}2"] = np.nan
            return cve_table

        for col in lag_feature_cols:
            cve_table[f"{col}2"] = cve_table[col].shift(-1)
        return cve_table.iloc[:-1].copy()

    lag_parts = [add_time_lag(group) for _, group in dt.groupby("cve_id", sort=False)]
    lag_dt = pd.concat(lag_parts, ignore_index=True)
    print(f"After Appending Next Time Period Variables: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

## Remove Short CVEs

In [9]:
if RUN_PREPROCESSING:
    cve_counts = lag_dt.groupby("cve_id").size()
    short_cve_ids = cve_counts[cve_counts <= 7].index
    lag_dt = lag_dt[~lag_dt["cve_id"].isin(short_cve_ids)].copy()
    print(f"After Removing Short CVEs: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

## Addressing Determinism and High Intercorrelation Among Features

In [10]:
if RUN_PREPROCESSING:
    selected_cols = [
        "cve_id", "start",
        "mis_link", "silence", "code_dev", "file",
        "mail_dev", "thread", "commit", "churn",
        "mis_link2", "silence2", "code_dev2", "file2",
        "mail_dev2", "thread2", "commit2", "churn2"
    ]
    lag_dt = lag_dt[selected_cols].copy()
    print(f"After Correlation/Determinism Pruning: {lag_dt.shape[0]} rows × {lag_dt.shape[1]} columns")

## Binarized CVE Indicators

In [11]:
if RUN_PREPROCESSING:
    cve_binarized = pd.get_dummies(
        lag_dt["cve_id"].astype("Int64").astype(str),
        prefix="b",
        dtype=int
    )

    binarized_lag_dt = pd.concat([
        lag_dt.drop(columns=["cve_id"]).reset_index(drop=True),
        cve_binarized.reset_index(drop=True)
    ], axis=1)
    print(f"After Binarize CVE ID: {binarized_lag_dt.shape[0]} rows × {binarized_lag_dt.shape[1]} columns")

## Add Null Features

## Keep only 5 null indicator features

In [12]:
if RUN_PREPROCESSING:
    rng = np.random.default_rng(SEED)
    null_df = pd.DataFrame(
        {col: rng.permutation(binarized_lag_dt[col].to_numpy()) for col in binarized_lag_dt.columns}
    )
    null_df.columns = [f"nv-{col}" for col in null_df.columns]

    null_non_indicator_cols = [c for c in null_df.columns if not c.startswith("nv-b_")]
    null_indicator_cols = [c for c in null_df.columns if c.startswith("nv-b_")]
    null_keep_cols = null_non_indicator_cols + null_indicator_cols[:5]
    null_df = null_df[null_keep_cols]

    processed_dt = pd.concat([binarized_lag_dt, null_df], axis=1)

    processed_dt.to_csv(DATA_PATH, index=False)
    binarized_lag_dt.to_csv(BINARIZED_DATA_PATH, index=False)

    print(f"✓ Saved null-variable dataset: {DATA_PATH} ({processed_dt.shape[0]} × {processed_dt.shape[1]})")
    print(f"✓ Saved non-null dataset: {BINARIZED_DATA_PATH} ({binarized_lag_dt.shape[0]} × {binarized_lag_dt.shape[1]})")

# Load dataset used for null variable causal search
data = load_data(DATA_PATH)
print(f"Dataset: {data.shape[0]} rows × {data.shape[1]} columns")

Dataset: 4870 rows × 138 columns


## Variable Analysis

Identify binary CVE indicators, continuous metrics, and null variables.

In [13]:
var_types = detect_variable_types(data)
print(f"Binary indicators: {len(var_types['cve_indicators'])}")
print(f"Continuous metrics: {len(var_types['metrics'])}")
print(f"Null variables: {len(var_types['null_variables'])}")

Binary indicators: 98
Continuous metrics: 17
Null variables: 23


# FGES Null Variable Search

Executes causal discovery with bootstrapping over the null-variable dataset.

In [14]:
# Execute algorithm
if ALGORITHM == "boss":
    results = run_boss_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT  # Toggle bootstrap output
    )
else:
    results = run_fges_analysis(
        data=data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT  # Toggle bootstrap output
    )

print(f"\nDiscovered: {results['n_nodes']} nodes, {results['n_edges']} edges")
print(f"Elapsed: {results['elapsed_time']:.1f}s")

AttributeError: Unable to export graph JSON. No compatible method found (tried get_json/getJson and GraphSaveLoadUtils.graphToJson).

---

# Deriving the 1 PNEF Threshold

In our causal search above, we introduced null features over multiple bootstrap runs to observe how often our causal search forms random edges (i.e. between our features and null features). We will use this information to derive a threshold, **1PNEF** (1st Percentile NoEdge Frequency), we can use in our final causal search.

## Graph Examination

We parse the Tetrad JSON graph output into tabular format: nodes, edgeset, and edge type probabilities.

The **edgeset** table contains the ensemble edge for each node pair. Because we performed multiple bootstrap runs, the probabilities represent the ensemble of all edges formed on each execution.

The **edge_type_probabilities** table shows the counts of each type of edge formed on each subgraph across all bootstrap runs.

In [ ]:
# Parse the JSON output from null variable causal search
null_graph = parse_graph(str(results['graph_json']))

print(f"Nodes: {len(null_graph['nodes'])}")
print(f"\nFirst 5 nodes:")
print(null_graph['nodes'].head())
print(f"\nEdgeset: {len(null_graph['edgeset'])} edges")
print(null_graph['edgeset'].head())
print(f"\nEdge type probabilities: {len(null_graph['edge_type_probabilities'])} entries")
print(null_graph['edge_type_probabilities'].head())

Nodes: 138

First 5 nodes:
  node_name
0  b_100433
1  b_100740
2  b_100742
3  b_102939
4  b_103864

Edgeset: 94 edges
  node1_name node2_name endpoint1 endpoint2   bold  highlighted properties  \
0   code_dev  code_dev2      TAIL     ARROW  False        False        NaN   
1      start   b_160705      TAIL     ARROW  False        False        NaN   
2    silence   silence2      TAIL     ARROW  False        False      dd;pl   
3    silence   mail_dev      TAIL     ARROW  False        False      dd;nl   
4      start    commit2      TAIL     ARROW  False        False      dd;nl   

   probability  
0     1.000000  
1     0.693069  
2     0.970297  
3     1.000000  
4     0.910891  

Edge type probabilities: 298 entries
  node1_name node2_name edge_type properties  probability
0   code_dev  code_dev2        ta        NaN     0.603960
1   code_dev  code_dev2        at        NaN     0.376238
2   code_dev  code_dev2        tt        NaN     0.019802
3      start   b_160705        ta        

## Deriving 1 PNEF

Our interest is to derive a threshold for the final causal search, using the information from this bootstrapped null feature causal search. By definition, edges formed between actual variables and random (null) features represent random edges.

We:
1. Subset the edgeset to contain only edges where at least one node is a null variable (nv-*)
2. Derive a `no_edge` probability by subtracting the probability from 1
3. Identify the 1st percentile value of the no_edge probability → the **1PNEF threshold**

This threshold tells us: given entirely random variables, causal links were formed between them up to X% of the time. In our final search, we only keep causal links that formed **more** than X% of the time.

In [ ]:
# Deriving 1 PNEF (R-equivalent explicit steps)
nv_edges = null_graph['edgeset'].copy()
is_node1_nv = nv_edges['node1_name'].astype(str).str.contains('nv-', regex=False)
is_node2_nv = nv_edges['node2_name'].astype(str).str.contains('nv-', regex=False)
nv_edges = nv_edges[is_node1_nv | is_node2_nv].copy()

# no_edge probability
nv_edges['no_edge'] = 1 - nv_edges['probability']

# 1st Percentile NoEdge Frequency (1PNEF)
pnef_1 = float(nv_edges['no_edge'].quantile(PNEF_PERCENTILE))

print(f"\n1PNEF Threshold: {pnef_1:.4f}")
print(f"Null-edge rows used: {len(nv_edges)}")
print("\nSample of null-variable edges:")
nv_edges.head(10)

Null variable edges: 1
1PNEF threshold (percentile=0.01): 0.4554
  → Random edges formed up to 54.5% of the time
  → Keep only edges with probability > 54.5%

1PNEF Threshold: 0.4554

Sample of null variable edges:


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
53,b_173733,nv-b_102939,TAIL,ARROW,False,False,pd;pl,0.544554,0.455446


---

# Non-Null Causal Search

With the threshold defined, we now proceed to the final causal search, which **does not include null features**. In this non-null feature causal search, we also specify domain knowledge to prohibit causal links that don't make sense temporally (e.g. features at 1-time-lag cannot cause features in the present).

## Domain Knowledge Causal Search without Null Variables

Remove null variable columns (nv-*) from the dataset, keeping only the original features and binary CVE indicators.

In [ ]:
# Domain Knowledge Causal Search without Null Variables
non_null_data = prepare_non_null_dataset(data, save_path=BINARIZED_DATA_PATH)
print(f"Prepared non-null dataset: {non_null_data.shape[0]} rows × {non_null_data.shape[1]} columns")
print(f"Saved to: {BINARIZED_DATA_PATH}")

Removed 23 null variable columns
Non-null dataset: 4870 rows × 115 columns
Saved to: binarized_variable_dt.csv


## Causal Search

Run the causal search on the non-null dataset with domain knowledge constraints. This search uses the same algorithm and bootstrap settings, but on the dataset **without** null features and **with** temporal knowledge constraints.

In [ ]:
# Run domain knowledge causal search on non-null dataset
if ALGORITHM == "boss":
    domain_results = run_boss_analysis(
        data=non_null_data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=NON_NULL_OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT
    )
else:
    domain_results = run_fges_analysis(
        data=non_null_data,
        knowledge_file=KNOWLEDGE_FILE,
        output_dir=NON_NULL_OUTPUT_DIR,
        penalty_discount=PENALTY_DISCOUNT,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
        suppress_output=not SHOW_BOOTSTRAP_OUTPUT
    )

print(f"\nDomain search discovered: {domain_results['n_nodes']} nodes, {domain_results['n_edges']} edges")
print(f"Elapsed: {domain_results['elapsed_time']:.1f}s")

BOSS completed: 115 nodes, 94 edges
Elapsed: 35.7s

Domain search discovered: 115 nodes, 94 edges
Elapsed: 35.7s


## Graph Examination

Parse the domain knowledge causal search JSON output into nodes, edgeset, and edge type probabilities.

In [ ]:
# Parse the domain search JSON output
domain_graph = parse_graph(str(domain_results['graph_json']))

print(f"Domain search nodes: {len(domain_graph['nodes'])}")
print(f"Domain search edges: {len(domain_graph['edgeset'])}")
print(f"\nEdgeset sample:")
domain_graph['edgeset'].head()


Domain search nodes: 115
Domain search edges: 94

Edgeset sample:


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability
0,code_dev,code_dev2,TAIL,ARROW,False,False,NaN,1.000000
1,start,b_160705,TAIL,ARROW,False,False,NaN,0.693069
2,silence,mail_dev,TAIL,ARROW,False,False,dd;nl,1.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;pl,0.960396
4,churn,commit2,TAIL,ARROW,False,False,NaN,0.811881


---

# Applying 1PNEF Threshold

## Applying 1PNEF Threshold

Edges which may have been formed at random are filtered here. We apply the 1PNEF threshold derived from the null variable search to the domain knowledge search results. Only edges whose `no_edge` probability is less than or equal to the 1PNEF threshold are kept.

In [ ]:
# Applying 1PNEF Threshold (R-equivalent explicit steps)
edges = domain_graph['edgeset'].copy()
edges['no_edge'] = 1 - edges['probability']
edges_1pnef = edges[edges['no_edge'] <= pnef_1].copy()

print(f"\nFiltered edges (1PNEF trimmed): {len(edges_1pnef)}")
edges_1pnef.head(20)

Edges before threshold: 94
Edges after 1PNEF threshold: 94
Removed: 0 edges (potentially random)

Filtered edges (1PNEF trimmed):


,node1_name,node2_name,endpoint1,endpoint2,bold,highlighted,properties,probability,no_edge
0,code_dev,code_dev2,TAIL,ARROW,False,False,NaN,1.000000,0.000000
1,start,b_160705,TAIL,ARROW,False,False,NaN,0.693069,0.306931
2,silence,mail_dev,TAIL,ARROW,False,False,dd;nl,1.000000,0.000000
3,silence,silence2,TAIL,ARROW,False,False,dd;pl,0.960396,0.039604
4,churn,commit2,TAIL,ARROW,False,False,NaN,0.811881,0.188119
...,...,...,...,...,...,...,...,...,...
89,file,mis_link,TAIL,ARROW,False,False,pd;nl,0.851485,0.148515
90,start,b_64339,TAIL,ARROW,False,False,pd;nl,0.960396,0.039604
91,silence,churn2,TAIL,ARROW,False,False,dd;pl,0.702970,0.297030
92,commit,code_dev,TAIL,ARROW,False,False,pd;nl,0.693069,0.306931


In [ ]:
# Save the 1PNEF-filtered edges
os.makedirs(NON_NULL_OUTPUT_DIR, exist_ok=True)
edges_1pnef.to_csv(f"{NON_NULL_OUTPUT_DIR}/edges_1pnef.csv", index=False)
print(f"✓ Saved filtered edges to {NON_NULL_OUTPUT_DIR}/edges_1pnef.csv")

✓ Saved filtered edges to boss_domain_results/edges_1pnef.csv


<!-- ---

# Results

With the final causal graph trimmed, we can now inspect it to draw conclusions. Causal graphs may form cycles and have undirected edges.

## Full Causal Graph 1-PNEF Trimmed

Interactive visualization of the full causal graph after applying the 1PNEF threshold.

Edge colors:
- **Black**: Directed edges (causal relationship)
- **Red**: Undirected edges (TAIL-TAIL, association without determined direction) -->

In [ ]:
# # Prepare edges for visualization
# viz_edges = prepare_graph_edges(edges_1pnef)

# # Plot the full causal graph
# full_graph = plot_causal_graph(
#     edges_df=viz_edges,
#     graph_nodes=domain_graph['nodes'],
#     title="Full Causal Graph (1PNEF Trimmed)",
#     output_html=f"{NON_NULL_OUTPUT_DIR}/causal_graph_full.html"
# )

# # Display in notebook (if pyvis available)
# if full_graph is not None:
#     full_graph.show(f"{NON_NULL_OUTPUT_DIR}/causal_graph_full.html")

Graph saved to: boss_domain_results/causal_graph_full.html
boss_domain_results/causal_graph_full.html


<!-- ## Sub-Graphs of Effort Variables and Parents

Focus on key effort variables and their neighboring causal structure in a smaller sub-graph. -->

In [ ]:
# # Plot sub-graph for effort variables and parents
# sub_graph = plot_subgraph(
#     edges_df=viz_edges,
#     nodes_of_interest=NODES_OF_INTEREST,
#     graph_nodes=domain_graph['nodes'],
#     title="Sub-Graphs of Effort Variables and Parents",
#     output_html=f"{NON_NULL_OUTPUT_DIR}/causal_graph_subgraph.html",
#     include_parents=True
#  )

# if sub_graph is not None:
#     sub_graph.show(f"{NON_NULL_OUTPUT_DIR}/causal_graph_subgraph.html")

Sub-graph: 10 nodes, 31 edges
Graph saved to: boss_domain_results/causal_graph_subgraph.html
boss_domain_results/causal_graph_subgraph.html


<!-- ## Cycle Detection

Check if the causal graph contains any cycles. Cycles indicate feedback loops in the causal structure. -->

In [ ]:
# # Detect cycles in the 1PNEF-trimmed graph
# cycles = find_cycles(viz_edges)
# print_cycle_report(cycles)

Found 4 cycle(s):
  Cycle 1 (length 3): thread2 → churn2 → churn → thread2
  Cycle 2 (length 4): thread2 → churn2 → churn → file2 → thread2
  Cycle 3 (length 3): thread2 → churn2 → mail_dev2 → thread2
  Cycle 4 (length 3): file2 → churn2 → churn → file2
